In [1]:
import zipfile

zip_ref = zipfile.ZipFile("sms+spam+collection.zip", "r")
zip_ref.extractall("sms_data")
zip_ref.close()

In [2]:
!pip install scikit-learn pandas numpy

In [4]:
import pandas as pd

df = pd.read_csv(
    "sms_data/SMSSpamCollection",
    sep="\t",
    names=["label", "message"]
)

df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [5]:
print(df.shape)
print(df["label"].value_counts())

(5572, 2)
label
ham     4825
spam     747
Name: count, dtype: int64


In [6]:
df["label"] = df["label"].map({"ham":0, "spam":1})

X = df["message"]
y = df["label"]

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english")

In [9]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

nb = MultinomialNB()
lr = LogisticRegression(max_iter=1000)
svm = LinearSVC()

In [10]:
from sklearn.ensemble import VotingClassifier

voting_hard = VotingClassifier(
    estimators=[("nb", nb), ("lr", lr), ("svm", svm)],
    voting="hard"
)

voting_soft = VotingClassifier(
    estimators=[("nb", nb), ("lr", lr)],
    voting="soft"
)

In [11]:
from sklearn.ensemble import StackingClassifier

stacking = StackingClassifier(
    estimators=[("nb", nb), ("lr", lr), ("svm", svm)],
    final_estimator=LogisticRegression()
)

In [12]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier

stump = DecisionTreeClassifier(max_depth=1)

ada = AdaBoostClassifier(
    estimator=stump,
    n_estimators=200
)

In [13]:
models = {
    "NaiveBayes": nb,
    "LogisticRegression": lr,
    "LinearSVM": svm,
    "VotingHard": voting_hard,
    "VotingSoft": voting_soft,
    "Stacking": stacking,
    "AdaBoost": ada
}

In [14]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
import numpy as np

skf = StratifiedKFold(n_splits=5)

results = []

for name, model in models.items():

    pipeline = Pipeline([
        ("tfidf", tfidf),
        ("model", model)
    ])

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=skf,
        scoring="f1"
    )

    results.append({
        "Model": name,
        "F1_mean": scores.mean(),
        "F1_std": scores.std()
    })

results_df = pd.DataFrame(results)

results_df

,Model,F1_mean,F1_std
0,NaiveBayes,0.862847,0.039089
1,LogisticRegression,0.812063,0.027663
2,LinearSVM,0.938948,0.005593
3,VotingHard,0.891453,0.027889
4,VotingSoft,0.851146,0.031208
5,Stacking,0.946334,0.004046
6,AdaBoost,0.687942,0.032958


In [15]:
results_df.to_csv("ensemble_comparison.csv", index=False)

In [16]:
pipeline = Pipeline([
    ("tfidf", tfidf),
    ("model", stacking)
])

pipeline.fit(X_train, y_train)

preds = pipeline.predict(X_test)

In [17]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, preds))
print(classification_report(y_test, preds))

[[962   4]
 [ 13 136]]
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       966
           1       0.97      0.91      0.94       149

    accuracy                           0.98      1115
   macro avg       0.98      0.95      0.97      1115
weighted avg       0.98      0.98      0.98      1115



In [18]:
output = pd.DataFrame({
    "MessageId": range(len(X_test)),
    "Actual": y_test.values,
    "Predicted": preds
})

output.to_csv("final_model_predictions.csv", index=False)

In [19]:
from google.colab import files

files.download("ensemble_comparison.csv")
files.download("final_model_predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>